# Learning Objectives

* [CUSTOMIZE] Build a real Pegasus workflow from a set of Python wrapper scripts and an Apptainer container.
* Generate and register input data, a container, and a workflow DAG through the Pegasus Python API.
* Visualize a workflow DAG before submitting it.
* Plan, submit, monitor, and pull statistics for a Pegasus workflow from a notebook.
* [CUSTOMIZE] Inspect the scientific outputs produced by a run.

# [CUSTOMIZE: Workflow Title]

[CUSTOMIZE] One paragraph: what the pipeline does, where it came from (paper, Nextflow/Snakemake
pipeline), and how it parallelizes. See `README.md` in this repository for the full pipeline description,
output listing, and conversion notes.

**Workflow Jobs:**
1. `[step1]` - [description]
2. `[step2]` - [description]
3. `[step3]` - [description]

**Prerequisites:** [Pegasus WMS](https://pegasus.isi.edu/) >= 5.0, [HTCondor](https://htcondor.org/) >= 10.2,
Python 3.8+ with `pyyaml`, and [Apptainer](https://apptainer.org/) — all assumed to already be available in
this notebook's environment.

## 1. Prepare Sample Input Data

[CUSTOMIZE] Say what the test data is, where it comes from, and what the script writes (inputs plus any
samplesheet). Run this on its own any time you just want fresh input data; it's independent of the rest of
this notebook. Delete this section if the workflow fetches its data at run time.

In [ ]:
!./prepare_test_data.sh

## 2. Build the Apptainer Container

The workflow's jobs run inside an Apptainer container (`Apptainer/My_Container.def`) that bundles
[CUSTOMIZE: tools]. Building it can take a few minutes; skip the rebuild if the `.sif` already exists.

In [ ]:
%%bash
if [ ! -f Apptainer/My_Container.sif ]; then
    apptainer build Apptainer/My_Container.sif Apptainer/My_Container.def
else
    echo "Container already built: Apptainer/My_Container.sif"
fi

## 3. Generate the Workflow

The following cell uses the `MyWorkflow` class from `workflow_generator.py` — the same generator used by the
command-line tool — to build the Pegasus workflow DAG (site, transformation, and replica catalogs) and
writes it to `workflow.yml`. Nothing about the workflow is defined here: change the pipeline in
`workflow_generator.py`, and both the notebook and the CLI pick it up.

`create_sites_catalog()` writes a self-contained site catalog for this machine's HTCondor pool (`local` +
`compute`). To run on a resource with a centrally hosted site catalog
([pegasus-site-catalogs](https://github.com/pegasushub/pegasus-site-catalogs/tree/main/conf), e.g.
`access-pegasus.yml`, `unity.yml`), set `HOSTED_SITE_CATALOG` instead — or set it once in `~/.pegasusrc`.

In [ ]:
import argparse
import sys
from pathlib import Path

sys.path.insert(0, str(Path(".").resolve()))

from workflow_generator import MyWorkflow  # [CUSTOMIZE] class (and input parser, if any)

DAGFILE = "workflow.yml"
EXEC_SITE = "compute"
HOSTED_SITE_CATALOG = None  # e.g. "unity.yml"; None = local HTCondor catalog below

# [CUSTOMIZE] The workflow-specific inputs the CLI would parse
ARGS = argparse.Namespace(items=["foo", "bar"])

workflow = MyWorkflow(dagfile=DAGFILE)  # [CUSTOMIZE] constructor arguments

print("Creating workflow properties...")
workflow.create_pegasus_properties(hosted_site_catalog=HOSTED_SITE_CATALOG)

if HOSTED_SITE_CATALOG is None:
    print("Creating execution sites...")
    workflow.create_sites_catalog(exec_site_name=EXEC_SITE)

print("Creating transformation catalog...")
workflow.create_transformation_catalog(exec_site_name=EXEC_SITE)

print("Creating replica catalog...")
workflow.create_replica_catalog()

print("Creating workflow DAG...")
workflow.create_workflow(ARGS)

workflow.write()
print("\nWorkflow has been generated!")

## 4. View the Generated Workflow DAG

Before submitting, visualize the workflow DAG using `pegasus-graphviz`. [CUSTOMIZE] Say what the graph
shows (per-sample chains, fan-in steps, ...).

In [ ]:
!pegasus-graphviz -f workflow.yml --output workflow.png

In [ ]:
from IPython.display import Image

Image(filename="workflow.png")

## 5. Plan and Submit the Workflow

The workflow will be planned and submitted for execution on the `compute` site defined in the site catalog
chosen above.

The CLI version of this generator (`workflow_generator.py`) writes the same workflow non-interactively and
prints the `pegasus-plan` command; it never submits by itself. Submitting is this explicit step.

In [ ]:
workflow.plan_submit(exec_site_name=EXEC_SITE)

## Workflow Status Monitoring

After successful submission, monitor the workflow status. The output shows job counts and idle/running/completed
states. `wait()` blocks (with a progress bar) until the workflow completes or fails.

In [ ]:
workflow.status()

In [ ]:
workflow.wait()

## 6. Statistics

Once the workflow completes, pull execution statistics from the Pegasus provenance database.

In [ ]:
workflow.statistics()

## 7. Examine the Results

The workflow stages the following final outputs to `output/`:

| Output | Description |
|--------|-------------|
| [CUSTOMIZE] `{sample}_result.png` | [description] |

In [ ]:
!ls -ltR output/

[CUSTOMIZE] Show one or two key results inline (a plot, a summary table).

In [ ]:
Image(filename="output/[CUSTOMIZE]_result.png")